# MMF Post-Processing & Evaluation — Reproducibility Notebook
Auto-generated by mmf-agent after interactive `/post-process-and-evaluate` session.

This notebook records the evaluation queries and best-model selection logic
and can be re-run to reproduce the same evaluation summary tables.

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
metric = "{metric}"

In [ ]:
eval_table = f"{catalog}.{schema}.{use_case}_evaluation_output"
score_table = f"{catalog}.{schema}.{use_case}_scoring_output"
nf_eval_table = f"{catalog}.{schema}.{use_case}_nf_evaluation_output"
nf_score_table = f"{catalog}.{schema}.{use_case}_nf_scoring_output"
fallback_table = f"{catalog}.{schema}.{use_case}_scoring_output_non_forecastable"
config_table = f"{catalog}.{schema}.{use_case}_pipeline_config"

# The strategy decides which non-forecastable outputs are merged into best_models.
# A missing config table means the series were never split: treat as 'include'.
try:
    rows = spark.sql(f"SELECT non_forecastable_strategy FROM {config_table} WHERE use_case = '{use_case}'").collect()
    strategy = (rows[0][0] if rows else None) or "include"
except Exception:
    strategy = "include"
print(f"Non-forecastable strategy: {strategy}")

required = [eval_table, score_table]
if strategy == "separate_job":
    required += [nf_eval_table, nf_score_table]
elif strategy == "fallback":
    required += [fallback_table]
for t in required:
    cnt = spark.sql(f"SELECT COUNT(*) AS cnt FROM {t}").collect()[0]["cnt"]
    print(f"{t}: {cnt:,} rows")
    assert cnt > 0, f"Table {t} is empty"

In [ ]:
best_forecastable_table = f"{catalog}.{schema}.{use_case}_best_models_forecastable"
best_table = f"{catalog}.{schema}.{use_case}_best_models"


def best_models_sql(eval_t, score_t, source):
    return f"""
WITH latest_run_per_model AS (
  -- MMF runs one job per model class, each writing its own run_id. Pick each model's
  -- most recent run_id (not a single global latest run) so all models compete head-to-head.
  -- A model's forecast and residuals share the same run_id, so this stays coherent for
  -- hierarchical reconciliation (which joins residuals per unique_id/model/run_id).
  SELECT model, run_id
  FROM (
    SELECT model, run_id,
           ROW_NUMBER() OVER (PARTITION BY model ORDER BY run_date DESC, run_id DESC) AS rn
    FROM (
      SELECT DISTINCT model, run_id, run_date
      FROM {eval_t}
    )
  )
  WHERE rn = 1
)
SELECT eval.unique_id, eval.model, eval.avg_metric, score.ds, score.y,
       '{source}' AS forecast_source, eval.run_id
FROM (
  SELECT unique_id, model, run_id, avg_metric,
         ROW_NUMBER() OVER (
           PARTITION BY unique_id
           ORDER BY
             avg_metric ASC,
             -- Tie-break: prefer the cheaper/simpler model on equal metrics (lower cost tier
             -- wins). ROW_NUMBER guarantees exactly one winning row per series.
             CASE
               WHEN model LIKE 'StatsForecast%' OR model LIKE 'SKTime%' THEN 1
               WHEN model LIKE 'MLForecast%'                            THEN 2
               WHEN model LIKE 'NeuralForecast%'                        THEN 3
               WHEN model LIKE 'Chronos%' OR model LIKE 'TimesFM%'
                    OR model LIKE 'FF%'                                 THEN 4
               ELSE 5
             END ASC,
             model ASC
         ) AS rank
  FROM (
    SELECT e.unique_id, e.model, e.run_id, AVG(e.metric_value) AS avg_metric
    FROM {eval_t} AS e
    INNER JOIN latest_run_per_model AS lr
      ON e.model = lr.model AND e.run_id = lr.run_id
    GROUP BY e.unique_id, e.model, e.run_id
    HAVING AVG(e.metric_value) IS NOT NULL
  )
) AS eval
INNER JOIN {score_t} AS score
  ON eval.unique_id = score.unique_id AND eval.model = score.model AND eval.run_id = score.run_id
WHERE eval.rank = 1
"""


spark.sql(f"CREATE OR REPLACE TABLE {best_forecastable_table} AS " + best_models_sql(eval_table, score_table, "main_pipeline"))

merged = f"""
CREATE OR REPLACE TABLE {best_table} AS
SELECT unique_id, model, avg_metric, ds, y, forecast_source, run_id
FROM {best_forecastable_table}
"""
if strategy == "separate_job":
    merged += "UNION ALL\nSELECT * FROM (" + best_models_sql(nf_eval_table, nf_score_table, "non_forecastable_pipeline") + ")"
elif strategy == "fallback":
    # Fallback series have no backtest, so no metric, dates or run_id.
    merged += f"""
UNION ALL
SELECT unique_id, model, NULL AS avg_metric, NULL AS ds, y, 'fallback' AS forecast_source,
       NULL AS run_id
FROM {fallback_table}
"""
spark.sql(merged)

print("Best model selection:")
spark.sql(f"SELECT forecast_source, COUNT(*) AS series_count FROM {best_table} GROUP BY forecast_source").show(truncate=False)

In [ ]:
best_table = f"{catalog}.{schema}.{use_case}_best_models"
eval_table = f"{catalog}.{schema}.{use_case}_evaluation_output"
summary_table = f"{catalog}.{schema}.{use_case}_evaluation_summary"
metric_col = f"avg_{metric}"

# Fallback series have no backtest, so their WAPE stays NULL.
nf_wape_branch = ""
if strategy == "separate_job":
    nf_wape_branch = f"""
  UNION ALL
  SELECT 'non_forecastable_pipeline' AS forecast_source, unique_id, model, run_id,
         AVG(
           AGGREGATE(
             TRANSFORM(ARRAYS_ZIP(forecast, actual), x -> ABS(x.actual - x.forecast)),
             CAST(0.0 AS DOUBLE), (acc, x) -> acc + x
           ) /
           NULLIF(AGGREGATE(TRANSFORM(actual, x -> ABS(x)), CAST(0.0 AS DOUBLE), (acc, x) -> acc + x), 0)
         ) AS wape
  FROM {nf_eval_table}
  GROUP BY unique_id, model, run_id"""

spark.sql(f"""
CREATE OR REPLACE TABLE {summary_table} AS
WITH series_wape AS (
  SELECT 'main_pipeline' AS forecast_source, unique_id, model, run_id,
         AVG(
           AGGREGATE(
             TRANSFORM(ARRAYS_ZIP(forecast, actual), x -> ABS(x.actual - x.forecast)),
             CAST(0.0 AS DOUBLE), (acc, x) -> acc + x
           ) /
           NULLIF(AGGREGATE(TRANSFORM(actual, x -> ABS(x)), CAST(0.0 AS DOUBLE), (acc, x) -> acc + x), 0)
         ) AS wape
  FROM {eval_table}
  GROUP BY unique_id, model, run_id
  {nf_wape_branch}
)
SELECT
    b.model,
    COUNT(*) AS wins_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS wins_pct,
    ROUND(AVG(b.avg_metric), 4) AS {metric_col},
    ROUND(AVG(w.wape), 4) AS avg_wape
FROM {best_table} AS b
LEFT JOIN series_wape AS w
  ON b.forecast_source = w.forecast_source AND b.unique_id = w.unique_id
     AND b.model = w.model AND b.run_id = w.run_id
GROUP BY b.model
ORDER BY wins_count DESC
""")

print("Model ranking:")
spark.sql(f"SELECT model, wins_count, wins_pct, {metric_col}, avg_wape FROM {summary_table}").show(truncate=False)

In [ ]:
best_table = f"{catalog}.{schema}.{use_case}_best_models"

overall = spark.sql(f"""
SELECT
  COUNT(DISTINCT unique_id) AS total_series,
  COUNT(DISTINCT model) AS distinct_winning_models,
  ROUND(AVG(avg_metric), 4) AS overall_avg_metric
FROM {best_table}
""").collect()[0]

print(f"Overall forecast quality:")
print(f"  Total series evaluated:    {overall['total_series']}")
print(f"  Distinct winning models:   {overall['distinct_winning_models']}")
print(f"  Overall avg {metric}:        {overall['overall_avg_metric']}")

print("\nBreakdown by forecast source:")
spark.sql(f"""
SELECT forecast_source, COUNT(DISTINCT unique_id) AS series_count,
       COUNT(DISTINCT model) AS distinct_models, ROUND(AVG(avg_metric), 4) AS avg_metric
FROM {best_table}
GROUP BY forecast_source
""").show(truncate=False)

print(f"\nWorst-performing series:")
spark.sql(f"""
SELECT unique_id, model, avg_metric, forecast_source
FROM {best_table}
WHERE avg_metric IS NOT NULL
ORDER BY avg_metric DESC
LIMIT 10
""").show(truncate=False)

In [ ]:
best_table = f"{catalog}.{schema}.{use_case}_best_models"
profile_table = f"{catalog}.{schema}.{use_case}_series_profile"

try:
    spark.sql(f"SELECT 1 FROM {profile_table} LIMIT 1")
    print("Cross-reference with series profiling:")
    spark.sql(f"""
    SELECT
        b.model,
        b.forecast_source,
        p.forecastability_class,
        COUNT(*) AS series_count,
        ROUND(AVG(b.avg_metric), 4) AS avg_metric
    FROM {best_table} b
    LEFT JOIN {profile_table} p ON b.unique_id = p.unique_id
    GROUP BY b.model, b.forecast_source, p.forecastability_class
    ORDER BY p.forecastability_class, b.forecast_source, avg_metric
    """).show(truncate=False)
except Exception:
    print("Series profile table not found — skipping cross-reference.")